In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from import_ipynb import NotebookFinder  # type: ignore
import importlib
import joblib
import os
import pandas as pd
from sklearn.manifold import TSNE
from sklearn.discriminant_analysis import StandardScaler


In [ ]:
# retrouver les dossiers
root = os.path.abspath(os.path.join(os.getcwd(), "../.."))
hyperparameter_dir = os.path.join(root, "pneumonia_knn", "documents", "model", "hyperparameter")
utils_dir = os.path.join(root, "pneumonia_knn","notebooks", "utils")

# charger les fichiers
# --- pneumonia_knn\notebooks\visualisation\accuracy_metrics.ipynb
spec_metrics = NotebookFinder().find_spec("metrics", [utils_dir])
metrics = importlib.util.module_from_spec(spec_metrics)
spec_metrics.loader.exec_module(metrics)

# --- pneumonia_knn\notebooks\utils
spec_print = NotebookFinder().find_spec("prints", [utils_dir])
prints = importlib.util.module_from_spec(spec_print)
spec_print.loader.exec_module(prints)

In [ ]:
# 1. Créer le pipeline
# Pipeline LDA solo
pipeline_lda = Pipeline([
    ('scaler', StandardScaler()),  # ← nécessaire car X_train brut
    ('lda', LDA(n_components=2)),
    ('knn', KNeighborsClassifier())
])

# Pipeline PCA + LDA
pipeline_pca_lda = Pipeline([
    ('lda', LDA(n_components=2)),  # ← pas de scaler, X_train_pca déjà normalisé
    ('knn', KNeighborsClassifier())
])

# 2. Définir les hyperparamètres
param_grid = {
    'knn__n_neighbors': [3, 5, 9],
    'knn__metric': ['euclidean', 'manhattan'],
    'knn__weights': ['distance'],
    'knn__algorithm': ['brute']
}

In [ ]:
def load_grid(file_name, X_train, y_train):
    pipeline = pipeline_pca_lda if 'pca' in file_name else pipeline_lda
    if os.path.exists(f'{hyperparameter_dir}/{file_name}'):
        # 2.5 Charger le GridSearchCV si il existe
        return joblib.load(f'{hyperparameter_dir}/{file_name}')
    else:
        # 2.5 Lancer GridSearchCV, sauvegarder le résultat et le retourner
        grid = GridSearchCV(
            estimator=pipeline,
            param_grid=param_grid,
            cv=2,
            scoring=metrics.get_scoring(),
            refit='recall',
            n_jobs=1,
            verbose=2
        )
        grid.fit(X_train, y_train)
        joblib.dump(grid, f'{hyperparameter_dir}/{file_name}')
        return grid

In [ ]:
def train_knn_lda(grid, X_test, y_test, X_test_2d_lda, X_test_2d_tsne):
    # 4. Récupérer le meilleur hyperparamètre
    knn_lda = grid.best_estimator_

    # 5. Prédire et évaluer les résultats
    y_pred_lda = knn_lda.predict(X_test)
    metrics_results_cross_val = pd.DataFrame(grid.cv_results_)

    # 6. Afficher les résultats
    prints.bold("Résultats liés à l'hyperparamètres tuning (GridSearchCV) :")
    metrics.show_all_types_of_metrics_from_hyperparameters_tuning(metrics_results_cross_val, 'Résultats GridSearchCV - KNN + LDA : toutes les métriques disponibles (GridSearchCV)')
    metrics.show_all_metrics_hyperparameters_tuning(metrics_results_cross_val, 'Résultats GridSearchCV - KNN + LDA : métriques principales (GridSearchCV)')

    prints.bold("Résultats liés à la prédiction :")
    metrics.show_plot_scatter_prediction(y_test, y_pred_lda, X_test_2d_lda, "LDA")
    metrics.show_plot_scatter_prediction(y_test, y_pred_lda, X_test_2d_tsne, "LDA + TSNE")
    metrics.show_plot_prediction(y_test, y_pred_lda, "LDA")

In [ ]:
# KNN avec LDA
def implementation_with_LDA(file_name,y_train, X_train, X_test, y_test):
    # 3. Charger le GridSearchCV
    grid = load_grid(file_name, X_train, y_train)

    # Créer les nouvelles données transformées par PCA pour les utiliser dans les autres algorithmes
    X_train_lda = grid.best_estimator_.named_steps['lda'].transform(X_train)
    X_test_lda = grid.best_estimator_.named_steps['lda'].transform(X_test)

    # Pas besoin de [:, :2] car LDA donne déjà 2 composantes (3 classes - 1)
    X_test_2d_lda = X_test_lda  # déjà en 2D
    tsne = TSNE(n_components=2, random_state=42)
    X_test_2d_tsne = tsne.fit_transform(X_test_lda)
    
    # 4. Entraîner le modèle KNN avec LDA
    train_knn_lda(grid, X_test, y_test, X_test_2d_lda, X_test_2d_tsne)